# Aggregate Weather Forecast Data

The weather forecast data is saved as parquet files for each day. In this notebook, the data is aggregated to one dataframe and then saved in one parquet file. Also data quality checks are made (duplicates and missing values).

In [ ]:
import pandas as pd
from pathlib import Path

### Load weather forecast data

In [2]:
base = Path.cwd() / "processed"

def load_daily(category: str) -> pd.DataFrame:
    files = sorted([p for p in (base / category).glob("*.parquet")])
    return pd.concat([pd.read_parquet(p) for p in files]).sort_index()

weather_fc_wind = load_daily("wind")
weather_fc_solar = load_daily("solar")

print("Data loaded:")
print(" - wind:", weather_fc_wind.shape)
print(" - solar:", weather_fc_solar.shape)

Data loaded:
 - wind: (26328, 6)
 - solar: (26328, 6)


### Convert to CET/CEST

In [3]:
weather_fc_wind = weather_fc_wind.tz_convert("Europe/Berlin")
weather_fc_solar = weather_fc_solar.tz_convert("Europe/Berlin")

### Check for duplicates

In [4]:
wind_dups = weather_fc_wind.loc[weather_fc_wind.index.duplicated(keep=False)]
solar_dups = weather_fc_solar.loc[weather_fc_solar.index.duplicated(keep=False)]

display("Duplicates wind:", wind_dups)
display("Duplicates solar:", solar_dups)

'Duplicates wind:'

,ssrd,tcc,t2m,sp,U100,rh2m
valid_time,,,,,,
2023-03-27 00:00:00+02:00,0.000000,87.046147,276.257185,99286.156840,7.814237,75.749404
2023-03-27 00:00:00+02:00,0.000000,64.627969,276.428051,99212.232378,8.442813,78.515634
2024-04-01 00:00:00+02:00,0.001839,76.897622,282.383304,98237.591908,4.649077,89.191025
2024-04-01 00:00:00+02:00,0.002016,85.446719,282.108439,98304.589407,4.577781,91.616744
2025-03-31 00:00:00+02:00,0.000307,70.830096,278.784812,100012.339852,10.081145,86.222526
2025-03-31 00:00:00+02:00,0.000290,72.467162,279.100065,100030.531075,9.907703,85.468597


'Duplicates solar:'

,ssrd,tcc,t2m,sp,U100,rh2m
valid_time,,,,,,
2023-03-27 00:00:00+02:00,0.000000,94.222622,276.696289,97418.389992,7.418201,84.294147
2023-03-27 00:00:00+02:00,0.000000,79.431889,276.971452,97298.274428,7.843325,85.945188
2024-04-01 00:00:00+02:00,0.000020,78.764265,282.831164,96424.294138,4.376136,85.390336
2024-04-01 00:00:00+02:00,0.000022,92.936774,282.762048,96545.404716,3.462679,87.678373
2025-03-31 00:00:00+02:00,0.000000,75.852624,278.532433,98355.714820,9.284414,84.513334
2025-03-31 00:00:00+02:00,0.000000,70.623421,278.795673,98368.363454,9.175010,85.485134


Remove duplicates (keep second entry, because first entry is forecast from two days before)

In [5]:
weather_fc_wind = weather_fc_wind[~weather_fc_wind.index.duplicated(keep="last")]
weather_fc_solar = weather_fc_solar[~weather_fc_solar.index.duplicated(keep="last")]

### Join solar and wind relevant columns

In [6]:
weather_fc = pd.concat([
    weather_fc_solar[["ssrd", "tcc", "t2m", "rh2m"]],
    weather_fc_wind[["U100", "sp"]],
], axis=1, join="inner")

# rename to clearify it's forecasts
weather_fc = weather_fc.rename(columns={"ssrd": "ssrd_fc",
                                        "tcc": "tcc_fc",
                                        "t2m": "t2m_fc",
                                        "rh2m": "rh2m_fc",
                                        "U100": "U100_fc",
                                        "sp": "sp_fc",})

### Check for missing timestamps

In [7]:
## check for missing timestamps
expected_index = pd.date_range(start=pd.Timestamp("2023-01-01 00:00", tz="Europe/Berlin"), end=pd.Timestamp("2025-12-31 23:00", tz="Europe/Berlin"), freq="h")
missing_timestamps = expected_index.difference(weather_fc.index)
print(missing_timestamps)

DatetimeIndex(['2023-10-29 23:00:00+01:00', '2024-10-27 23:00:00+01:00',
               '2025-10-26 23:00:00+01:00'],
              dtype='datetime64[ns, Europe/Berlin]', freq=None)


Three missing values because of the switch from daylight saving time to standard time -> linear imputation

In [8]:
weather_fc_imputed = weather_fc.sort_index().reindex(expected_index)
weather_fc_imputed = weather_fc_imputed.interpolate(method="linear")

### Save aggregated data

In [9]:
weather_fc_imputed.to_parquet("weather_forecast.parquet")